# Tools and Tool Calling Loop

In [11]:
from langchain.tools import tool
import requests
import json

In [18]:
res = requests.get(f"https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&ids=bitcoin")
data = res.text
data

'[{"id":"bitcoin","symbol":"btc","name":"Bitcoin","image":"https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400","current_price":80026,"market_cap":1608955875616,"market_cap_rank":1,"fully_diluted_valuation":1608955875616,"total_volume":34803041212,"high_24h":81004,"low_24h":78152,"price_change_24h":1783.48,"price_change_percentage_24h":2.27667,"market_cap_change_24h":35857585502,"market_cap_change_percentage_24h":2.27942,"circulating_supply":20105365.0,"total_supply":20105365.0,"max_supply":21000000.0,"ath":126080,"ath_change_percentage":-36.52739,"ath_date":"2025-10-06T10:57:42.000Z","atl":67.81,"atl_change_percentage":117917.07114,"atl_date":"2013-07-05T16:00:00.000Z","roi":null,"last_updated":"2026-11-05T12:31:55.000Z"}]'

In [19]:
@tool
def getCryptoPrice(coin: str) -> str:
    """Get the price of crypto currency of a given coin"""
    res = requests.get(f"https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&ids={coin}")
    data  = res.text
    return data

In [20]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3.8-27b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.5', 'langchain': '1.4.9'}}, client=<groq.resources.chat.completions.Completions object at 0x000001E9EEB001A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E9EEB00C20>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [21]:
model.invoke("Hello")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 13, 'total_tokens': 23, 'completion_time': 0.020003943, 'completion_tokens_details': None, 'prompt_time': 0.000834166, 'prompt_tokens_details': None, 'queue_time': 0.04827137, 'total_time': 0.020838109}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a1abb5-c177-7602-b965-1bb08c0bd83f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 10, 'total_tokens': 23})

In [22]:
model = model.bind_tools([getCryptoPrice])

In [23]:
response = model.invoke("What is the price of bitcoin?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'hhr9pfc9j', 'function': {'arguments': '{"coin":"bitcoin"}', 'name': 'getCryptoPrice'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 282, 'total_tokens': 309, 'completion_time': 0.069756307, 'completion_tokens_details': None, 'prompt_time': 0.019252042, 'prompt_tokens_details': None, 'queue_time': 0.056425383, 'total_time': 0.089008349}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1abb8-728d-75c8-800c-3fafe11573d6-0' tool_calls=[{'name': 'getCryptoPrice', 'args': {'coin': 'bitcoin'}, 'id': 'hhr9pfc9j', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 282, 'output_tokens': 27, 'total_tokens': 309}
Tool: getCryptoPrice
Args: {'coin': 'bitcoin'}


Tool Execution loops

In [26]:
messages = [
    {
        "role": "user",
        "content": "What is the price of Bitcoin?"
    }
]

ai_msg = model.invoke(messages)

messages.append(ai_msg)

In [27]:
for tool_call in ai_msg.tool_calls:
    tool_result = getCryptoPrice.invoke(tool_call)
    messages.append(tool_result)

In [28]:
response = model.invoke(messages)
print(response.text)

The current price of Bitcoin is **$79,548**.

Here are a few additional details:
*   **24h Change:** +1.79% (up $1,402.06)
*   **24h High:** $81,004
*   **24h Low:** $78,152
*   **Market Cap:** ~$1.60 Trillion
